In [ ]:
from datetime import datetime
import pandas as pd
import yaml as yml
import pycountry
from plotly import express as px

from emu_renewal.inputs import BASE_PATH, get_standard_priors
from emu_renewal.inputs import get_worldbank_national_pop, get_country_mobility, get_standard_targets, get_euro_var_inputs

pd.options.plotting.backend = "plotly"

In [ ]:
# Standard inputs
country = "Spain"
analysis_start = datetime(2020, 9, 1)
analysis_end = datetime(2021, 6, 15)
init_duration = 50
var_target_start_date = datetime(2021, 1, 1)
var_target_end_date = datetime(2021, 4, 1)
seed_duration = 10
analysis_to_data_delay = 14
date_format = "%d/%m/%Y"

In [ ]:
countries = ["BE", "CZ", "DE", "DE", "ES", "FI", "FR", "IT", "NL", "NO", "PL", "PT", "SE"]

In [ ]:
path = BASE_PATH / "preview_plots"
for country in countries:
    iso3 = pycountry.countries.lookup(country).alpha_3
    pop = get_worldbank_national_pop(iso3)
    mobility = get_country_mobility(iso3)
    cases_target, hosp_target, deaths_target, seroprev_target, init_data = get_standard_targets(country, analysis_start, analysis_end, init_duration, analysis_to_data_delay)
    strains = ["eu", "alpha"]
    var_target, seed_times = get_euro_var_inputs(country, strains, analysis_start, seed_duration, var_target_start_date, var_target_end_date)

    text = f"{country}\nISO3: {iso3}\npopulation: {pop / 1e6} million\n"
    text += f"analysis start: {datetime.strftime(analysis_start, date_format)}\n"
    text += f"analysis end: {datetime.strftime(analysis_end, date_format)}\n"
    text += f"alpha seed start: {datetime.strftime(seed_times[0][0], date_format)}\n"
    text += f"alpha seed end: {datetime.strftime(seed_times[0][1], date_format)}\n"
    open(BASE_PATH / f"preview_text/{country}_preview_text.txt", "w").write(text)

    mob_plot = mobility.plot(title=f"mobility {country}")
    mob_plot.write_image(path / f"mob_plot_{country}.png")
    cases_plot = px.scatter(cases_target, title=f"cases {country}")
    cases_plot.write_image(path / f"cases_plot_{country}.png")
    hosp_plot = px.scatter(hosp_target, title=f"hospital admissions {country}")
    hosp_plot.write_image(path / f"hosp_plot_{country}.png")
    sero_plot = px.scatter(seroprev_target, title=f"seroprevalence {country}")
    sero_plot.write_image(path / f"sero_plot_{country}.png")
    var_plot = px.scatter(var_target, title=f"variant proportions {country}")
    var_plot.write_image(path / f"var_plot_{country}.png")